# 📊 Data Visualization Project - E-Commerce Sales Dashboard -- CodeAlpha Task3

In [19]:
"""
E-commerce Sales Analytics Dashboard
CodeAlpha Data Analytics Internship - Task 3: Data Visualization
Author: [Sri Venkata Satyanarayana Gattu]
Date: [25-10-2025]

This script creates interactive visualizations for e-commerce sales data analysis.
All visualizations are interactive and can be explored by hovering, zooming, and panning.


"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIG - set your dataset path here (CSV or XLSX). Leave as None to use demo data.
# =============================================================================
file_path = '/content/data.csv'  # e.g. 'data.csv' or 'Online Retail.xlsx' ; set to None to use sample/demo data

In [2]:
# =============================================================================
# LOAD DATA (CSV or Excel) WITH SAFE FALLBACK TO DEMO DATA
# =============================================================================
def load_data(path):
    if path is None:
        return None
    try:
        if path.lower().endswith('.csv'):
            return pd.read_csv(path)
        elif path.lower().endswith(('.xls', '.xlsx')):
            return pd.read_excel(path)
        else:
            print(f"Unsupported extension for {path}. Using demo data.")
            return None
    except Exception as e:
        print(f"Failed to load {path}: {e}\nUsing demo data instead.")
        return None

df = load_data(file_path)

# If no real dataset provided or loading failed -> generate sample demo data
if df is None:
    import numpy as np
    from datetime import datetime, timedelta

    np.random.seed(42)
    dates = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')
    n_records = len(dates) * 10

    df = pd.DataFrame({
        'Date': np.random.choice(dates, n_records),
        'Order_ID': [f'ORD{i:06d}' for i in range(n_records)],
        'Product_Category': np.random.choice(['Electronics', 'Clothing', 'Home & Garden',
                                             'Sports', 'Books', 'Toys'], n_records),
        'Product_Name': [f'Product_{i}' for i in np.random.choice(range(100), n_records)],
        'Quantity': np.random.randint(1, 10, n_records),
        'Unit_Price': np.random.uniform(10, 500, n_records).round(2),
        'Customer_ID': [f'CUST{i:05d}' for i in np.random.choice(range(500), n_records)],
        'Region': np.random.choice(['North America', 'Europe', 'Asia', 'South America', 'Australia'],
                                  n_records),
        'Payment_Method': np.random.choice(['Credit Card', 'PayPal', 'Debit Card', 'Bank Transfer'],
                                          n_records)
    })
    df['Total_Amount'] = (df['Quantity'] * df['Unit_Price']).round(2)
    df['Date'] = pd.to_datetime(df['Date'])
    print("Demo sample data created.")
else:
    # If real dataset loaded, attempt to standardize column names and compute Total_Amount if missing
    # Try to infer date and numeric columns
    print(f"Loaded dataset from: {file_path}")
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    if 'Total_Amount' not in df.columns and {'Quantity', 'Unit_Price'}.issubset(df.columns):
        df['Total_Amount'] = (df['Quantity'] * df['Unit_Price']).round(2)
    # If Order_ID missing, create synthetic IDs for grouping purposes
    if 'Order_ID' not in df.columns:
        df['Order_ID'] = [f'ORD{i:06d}' for i in range(len(df))]

print("=" * 80)
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("=" * 80)

Failed to load /content/data.csv: 'utf-8' codec can't decode byte 0xa3 in position 79780: invalid start byte
Using demo data instead.
Demo sample data created.
Dataset Shape: (3650, 10)
Columns: ['Date', 'Order_ID', 'Product_Category', 'Product_Name', 'Quantity', 'Unit_Price', 'Customer_ID', 'Region', 'Payment_Method', 'Total_Amount']


In [3]:
# =============================================================================
# DATA PREPROCESSING
# =============================================================================
# Ensure Date column exists and drop rows with invalid dates
if 'Date' not in df.columns or df['Date'].isna().all():
    # If date not available, create synthetic dates to allow time-based visuals
    print("Warning: 'Date' column missing or invalid. Generating synthetic dates for visualization.")
    df['Date'] = pd.date_range(start='2023-01-01', periods=len(df), freq='D')

df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.day_name()

# Sort by date
df = df.sort_values('Date').reset_index(drop=True)

print("\n📊 Data preprocessing completed!")


📊 Data preprocessing completed!


In [4]:
# =============================================================================
# AGGREGATIONS NEEDED BY MULTIPLE VISUALIZATIONS / STATS
# =============================================================================
# Month ordering for consistent plots
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

monthly_revenue = df.groupby('Month_Name', as_index=False)['Total_Amount'].sum()
monthly_revenue['Month_Name'] = pd.Categorical(monthly_revenue['Month_Name'],
                                               categories=month_order, ordered=True)
monthly_revenue = monthly_revenue.sort_values('Month_Name').reset_index(drop=True)

monthly_orders = df.groupby('Month_Name', as_index=False)['Order_ID'].count()
monthly_orders['Month_Name'] = pd.Categorical(monthly_orders['Month_Name'],
                                              categories=month_order, ordered=True)
monthly_orders = monthly_orders.sort_values('Month_Name').reset_index(drop=True)

monthly_avg = df.groupby('Month_Name', as_index=False)['Total_Amount'].mean()
monthly_avg['Month_Name'] = pd.Categorical(monthly_avg['Month_Name'],
                                           categories=month_order, ordered=True)
monthly_avg = monthly_avg.sort_values('Month_Name').reset_index(drop=True)

daily_sales = df.groupby('Date', as_index=False)['Total_Amount'].sum()

category_performance = df.groupby('Product_Category', as_index=False).agg({
    'Total_Amount': 'sum',
    'Quantity': 'sum',
    'Order_ID': 'count'
})
category_performance.columns = ['Category', 'Total_Sales', 'Quantity_Sold', 'Number_of_Orders']
category_performance = category_performance.sort_values('Total_Sales', ascending=False).reset_index(drop=True)

regional_sales = df.groupby('Region', as_index=False)['Total_Amount'].sum().sort_values('Total_Amount', ascending=False).reset_index(drop=True)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_sales = df.groupby('Weekday', as_index=False)['Total_Amount'].sum()
weekday_sales['Weekday'] = pd.Categorical(weekday_sales['Weekday'], categories=weekday_order, ordered=True)
weekday_sales = weekday_sales.sort_values('Weekday').reset_index(drop=True)

product_sales = df.groupby('Product_Name', as_index=False)['Total_Amount'].sum()
top_products = product_sales.nlargest(10, 'Total_Amount').sort_values('Total_Amount').reset_index(drop=True)

payment_distribution = df.groupby('Payment_Method', as_index=False)['Total_Amount'].sum()

customer_by_region = df.groupby('Region', as_index=False)['Customer_ID'].nunique()

In [5]:
# =============================================================================
# VISUALIZATION 1: Monthly Revenue Trend (Interactive Area Chart)
# =============================================================================
print("\n🎨 Creating Visualization 1: Monthly Revenue Trend...")

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=monthly_revenue['Month_Name'],
    y=monthly_revenue['Total_Amount'],
    mode='lines+markers',
    name='Revenue',
    line=dict(color='#3b82f6', width=3),
    marker=dict(size=8, color='#3b82f6'),
    fill='tozeroy',
    fillcolor='rgba(59, 130, 246, 0.25)',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.2f}<extra></extra>'
))
fig1.update_layout(
    title='📈 Monthly Revenue Trend',
    xaxis_title='Month',
    yaxis_title='Revenue ($)',
    template='plotly_dark',
    hovermode='x unified',
    height=500,
    showlegend=True
)
fig1.show()
print("✅ Visualization 1 created successfully!")


🎨 Creating Visualization 1: Monthly Revenue Trend...


✅ Visualization 1 created successfully!


In [6]:
# =============================================================================
# VISUALIZATION 2: Product Category Performance (Interactive Bar Chart)
# =============================================================================
print("\n🎨 Creating Visualization 2: Product Category Performance...")

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=category_performance['Category'],
    y=category_performance['Total_Sales'],
    marker_color=['#8b5cf6', '#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#ec4899'][:len(category_performance)],
    text=category_performance['Total_Sales'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Sales: $%{y:,.2f}<br>Orders: %{customdata}<extra></extra>',
    customdata=category_performance['Number_of_Orders']
))
fig2.update_layout(
    title='🏆 Product Category Performance',
    xaxis_title='Category',
    yaxis_title='Total Sales ($)',
    template='plotly_dark',
    height=500,
    showlegend=False
)
fig2.show()
print("✅ Visualization 2 created successfully!")


🎨 Creating Visualization 2: Product Category Performance...


✅ Visualization 2 created successfully!


In [7]:
# =============================================================================
# VISUALIZATION 3: Regional Sales Distribution (Interactive Pie Chart)
# =============================================================================
print("\n🎨 Creating Visualization 3: Regional Sales Distribution...")

fig3 = go.Figure(data=[go.Pie(
    labels=regional_sales['Region'],
    values=regional_sales['Total_Amount'],
    hole=0.4,
    marker=dict(colors=['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6'][:len(regional_sales)]),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Sales: $%{value:,.2f}<br>Percentage: %{percent}<extra></extra>'
)])
fig3.update_layout(
    title='🌍 Sales Distribution by Region',
    template='plotly_dark',
    height=500,
    showlegend=True
)
fig3.show()
print("✅ Visualization 3 created successfully!")


🎨 Creating Visualization 3: Regional Sales Distribution...


✅ Visualization 3 created successfully!


In [8]:
# =============================================================================
# VISUALIZATION 4: Daily Sales Pattern (Interactive Line Chart)
# =============================================================================
print("\n🎨 Creating Visualization 4: Daily Sales Pattern...")

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=daily_sales['Date'],
    y=daily_sales['Total_Amount'],
    mode='lines',
    name='Daily Sales',
    line=dict(color='#10b981', width=2),
    hovertemplate='<b>Date: %{x|%Y-%m-%d}</b><br>Sales: $%{y:,.2f}<extra></extra>'
))
fig4.update_layout(
    title='📅 Daily Sales Pattern',
    xaxis_title='Date',
    yaxis_title='Sales ($)',
    template='plotly_dark',
    hovermode='x unified',
    height=500,
    showlegend=True
)
fig4.show()
print("✅ Visualization 4 created successfully!")


🎨 Creating Visualization 4: Daily Sales Pattern...


✅ Visualization 4 created successfully!


In [9]:
# =============================================================================
# VISUALIZATION 5: Sales by Weekday (Interactive Bar Chart)
# =============================================================================
print("\n🎨 Creating Visualization 5: Sales by Weekday...")

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    x=weekday_sales['Weekday'],
    y=weekday_sales['Total_Amount'],
    marker_color=['#ef4444', '#f59e0b', '#10b981', '#3b82f6', '#8b5cf6', '#ec4899', '#f97316'],
    text=weekday_sales['Total_Amount'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Sales: $%{y:,.2f}<extra></extra>'
))
fig5.update_layout(
    title='📊 Sales Performance by Day of Week',
    xaxis_title='Day of Week',
    yaxis_title='Total Sales ($)',
    template='plotly_dark',
    height=500,
    showlegend=False
)
fig5.show()
print("✅ Visualization 5 created successfully!")


🎨 Creating Visualization 5: Sales by Weekday...


✅ Visualization 5 created successfully!


In [10]:
# =============================================================================
# VISUALIZATION 6: Top 10 Products (Interactive Horizontal Bar Chart)
# =============================================================================
print("\n🎨 Creating Visualization 6: Top 10 Products...")

fig6 = go.Figure()
fig6.add_trace(go.Bar(
    x=top_products['Total_Amount'],
    y=top_products['Product_Name'],
    orientation='h',
    marker_color='#f59e0b',
    text=top_products['Total_Amount'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Sales: $%{x:,.2f}<extra></extra>'
))
fig6.update_layout(
    title='🏅 Top 10 Best-Selling Products',
    xaxis_title='Total Sales ($)',
    yaxis_title='Product',
    template='plotly_dark',
    height=600,
    showlegend=False
)
fig6.update_yaxes(tickfont=dict(size=10))
fig6.show()
print("✅ Visualization 6 created successfully!")


🎨 Creating Visualization 6: Top 10 Products...


✅ Visualization 6 created successfully!


In [11]:
# =============================================================================
# VISUALIZATION 7: Payment Method Distribution (Interactive Donut Chart)
# =============================================================================
print("\n🎨 Creating Visualization 7: Payment Method Distribution...")

fig7 = go.Figure(data=[go.Pie(
    labels=payment_distribution['Payment_Method'],
    values=payment_distribution['Total_Amount'],
    hole=0.5,
    # colors list truncated / applied based on number of payment methods
    marker=dict(colors=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'][:len(payment_distribution)]),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Amount: $%{value:,.2f}<br>Percentage: %{percent}<extra></extra>'
)])
fig7.update_layout(
    title='💳 Payment Method Distribution',
    template='plotly_dark',
    height=500,
    annotations=[dict(text='Payment<br>Methods', x=0.5, y=0.5, font_size=20, showarrow=False)]
)
fig7.show()
print("✅ Visualization 7 created successfully!")


🎨 Creating Visualization 7: Payment Method Distribution...


✅ Visualization 7 created successfully!


In [12]:
# =============================================================================
# VISUALIZATION 8: Quantity vs Revenue Scatter Plot
# =============================================================================
print("\n🎨 Creating Visualization 8: Quantity vs Revenue Analysis...")

fig8 = px.scatter(
    df,
    x='Quantity',
    y='Total_Amount',
    color='Product_Category' if 'Product_Category' in df.columns else None,
    size='Total_Amount',
    hover_data=['Product_Name', 'Region'] if {'Product_Name', 'Region'}.issubset(df.columns) else None,
    title='📦 Quantity vs Revenue Analysis',
    labels={'Quantity': 'Quantity Sold', 'Total_Amount': 'Revenue ($)'},
    template='plotly_dark',
    height=600
)
fig8.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color='white')))
fig8.show()
print("✅ Visualization 8 created successfully!")


🎨 Creating Visualization 8: Quantity vs Revenue Analysis...


✅ Visualization 8 created successfully!


In [13]:
# =============================================================================
# VISUALIZATION 9: Multi-Metric Dashboard (Subplots)
# =============================================================================
print("\n🎨 Creating Visualization 9: Comprehensive Dashboard...")

fig9 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Orders', 'Average Order Value',
                    'Sales Trend', 'Customer Distribution'),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'pie'}]]
)

# Subplot 1: Monthly Orders
fig9.add_trace(
    go.Bar(x=monthly_orders['Month_Name'], y=monthly_orders['Order_ID'],
           marker_color='#3b82f6', name='Orders'),
    row=1, col=1
)

# Subplot 2: Average Order Value
fig9.add_trace(
    go.Scatter(x=monthly_avg['Month_Name'], y=monthly_avg['Total_Amount'],
               mode='lines+markers', marker_color='#10b981', name='Avg Value'),
    row=1, col=2
)

# Subplot 3: Sales Trend (daily)
fig9.add_trace(
    go.Scatter(x=daily_sales['Date'], y=daily_sales['Total_Amount'],
               mode='lines', line=dict(color='#f59e0b', width=2), name='Daily Sales'),
    row=2, col=1
)

# Subplot 4: Customer Distribution (unique customers by region)
fig9.add_trace(
    go.Pie(labels=customer_by_region['Region'], values=customer_by_region['Customer_ID'],
           name='Customers'),
    row=2, col=2
)

fig9.update_layout(
    title_text='🎯 Comprehensive Sales Dashboard',
    template='plotly_dark',
    height=900,
    showlegend=True
)
fig9.show()
print("✅ Visualization 9 created successfully!")


🎨 Creating Visualization 9: Comprehensive Dashboard...


✅ Visualization 9 created successfully!


In [14]:
# =============================================================================
# VISUALIZATION 10: Heatmap - Sales by Category and Region
# =============================================================================
print("\n🎨 Creating Visualization 10: Category-Region Heatmap...")

heatmap_data = df.pivot_table(
    values='Total_Amount',
    index='Product_Category' if 'Product_Category' in df.columns else 'Product_Name',
    columns='Region' if 'Region' in df.columns else 'Month_Name',
    aggfunc='sum',
    fill_value=0
)

fig10 = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='Viridis',
    hovertemplate='Category: %{y}<br>Region: %{x}<br>Sales: $%{z:,.2f}<extra></extra>'
))
fig10.update_layout(
    title='🔥 Sales Heatmap: Category vs Region',
    xaxis_title='Region',
    yaxis_title='Product Category',
    template='plotly_dark',
    height=600
)
fig10.show()
print("✅ Visualization 10 created successfully!")


🎨 Creating Visualization 10: Category-Region Heatmap...


✅ Visualization 10 created successfully!


In [15]:
# =============================================================================
# KEY INSIGHTS & SUMMARY STATISTICS
# =============================================================================
print("\n" + "=" * 80)
print("📊 KEY INSIGHTS & SUMMARY STATISTICS")
print("=" * 80)

total_revenue = df['Total_Amount'].sum()
total_orders = df['Order_ID'].nunique()
total_customers = df['Customer_ID'].nunique() if 'Customer_ID' in df.columns else df['Order_ID'].nunique()
avg_order_value = df.groupby('Order_ID')['Total_Amount'].sum().mean()
total_quantity_sold = df['Quantity'].sum() if 'Quantity' in df.columns else None

print(f"\n💰 Total Revenue: ${total_revenue:,.2f}")
print(f"📦 Total Orders: {total_orders:,}")
print(f"👥 Total Customers (unique IDs): {total_customers:,}")
print(f"💵 Average Order Value: ${avg_order_value:,.2f}")
if total_quantity_sold is not None:
    print(f"📊 Total Products Sold: {total_quantity_sold:,}")

# Top performing category (if available)
if not category_performance.empty:
    top_category = category_performance.iloc[0]
    print(f"\n🏆 Top Category: {top_category['Category']} (${top_category['Total_Sales']:,.2f})")

# Top performing region (if available)
if not regional_sales.empty:
    top_region = regional_sales.iloc[0]
    print(f"🌍 Top Region: {top_region['Region']} (${top_region['Total_Amount']:,.2f})")

# Best day by revenue (if available)
if not weekday_sales.empty:
    best_day = weekday_sales.loc[weekday_sales['Total_Amount'].idxmax()]
    print(f"📅 Best Day: {best_day['Weekday']} (${best_day['Total_Amount']:,.2f})")

# Growth trend (first vs last month) - safe guard when first_month_sales is zero
if not monthly_revenue.empty:
    first_month_sales = monthly_revenue.iloc[0]['Total_Amount']
    last_month_sales = monthly_revenue.iloc[-1]['Total_Amount']
    if first_month_sales == 0:
        growth_rate = float('inf') if last_month_sales > 0 else 0.0
    else:
        growth_rate = ((last_month_sales - first_month_sales) / first_month_sales) * 100
    if growth_rate == float('inf'):
        growth_str = "∞ (first month sales = 0)"
    else:
        growth_str = f"{growth_rate:.1f}%"
    print(f"📈 Revenue Growth (first -> last month): {growth_str}")

print("\n" + "=" * 80)


📊 KEY INSIGHTS & SUMMARY STATISTICS

💰 Total Revenue: $4,667,691.50
📦 Total Orders: 3,650
👥 Total Customers (unique IDs): 500
💵 Average Order Value: $1,278.82
📊 Total Products Sold: 18,364

🏆 Top Category: Home & Garden ($808,418.51)
🌍 Top Region: Europe ($998,531.02)
📅 Best Day: Thursday ($728,671.73)
📈 Revenue Growth (first -> last month): -2.1%

